# Entity-swap (full) factor sweep — all relations

This Colab notebook runs the **full** entity-swap steering experiment (suppress the source Output CLT features **and** inject the donor Output CLT features) across **all 10 BATS relations** on the labeled "ours" summary graphs.

It sweeps two factors:

- **source suppression factor** over `0, -1, -2` (negation coefficients).
- **donor injection factor** over `1, 2, 4` (addition coefficients).

For each relation it samples 50 ordered source→donor pairs (`random_state=42`) and evaluates every (source_factor, donor_factor) cell, i.e. 3×3 = 9 cells per relation. It loads the replacement model once with the matching `mntss/clt-gemma-2-2b-426k` transcoder set and reports Top-1 / Top-5 hit rates, success, Δp_source, Δp_donor per relation and aggregated across relations.

In [1]:
import shutil
import subprocess

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("nvidia-smi not found; CPU will be slow for entity-swap interventions.")

Sun Jun 28 18:40:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.144.03             Driver Version: 550.144.03     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        Off |   00000000:3B:00.0 Off |                  Off |
|  0%   45C    P8             12W /  450W |       2MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/IamKrill1n/circuit_tracer_mod.git"
REPO_NAME = "circuit_tracer_mod"
REPO_REF = "clean_up"


def find_repo_root() -> Path | None:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    candidates.extend([Path("/home/tu/circuit_tracer_mod"), Path("/content") / REPO_NAME])
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (candidate / "summarization").is_dir():
            return candidate
    return None


REPO_ROOT = find_repo_root()
clone_parent = Path("/content") if Path("/content").exists() else Path.cwd()
if REPO_ROOT is None:
    subprocess.run(["git", "clone", REPO_URL, str(clone_parent / REPO_NAME)], check=True)
    REPO_ROOT = clone_parent / REPO_NAME

# Only fetch/checkout when this is a cloned Colab checkout, not the local working copy.
if REPO_ROOT == clone_parent / REPO_NAME and (REPO_ROOT / ".git").exists():
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_ROOT, check=True)
    subprocess.run(["git", "checkout", REPO_REF], cwd=REPO_ROOT, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", REPO_REF], cwd=REPO_ROOT, check=True)

print("repo root:", REPO_ROOT)
print(subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_ROOT, text=True).strip())

repo root: /home/tu/circuit_tracer_mod
c75719f


In [3]:
# Make repo imports work in this kernel, installing editable only on a fresh /content runtime.
repo_root_str = str(REPO_ROOT)
if repo_root_str not in sys.path:
    sys.path.insert(0, repo_root_str)

try:
    import circuit_tracer  # noqa: F401
except ModuleNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT)], check=True)
    if repo_root_str not in sys.path:
        sys.path.insert(0, repo_root_str)

from eval.eval_entity_swap import build_parser, run_entity_swap  # noqa: F401

# Sanity-check that this checkout exposes the negation/addition coefficient sweeps we rely on.
parser = build_parser()
dests = {a.dest for a in parser._actions}
assert {"negation_coefficients", "addition_coefficients", "relations"} <= dests, dests
print("entity-swap eval is importable; coefficient sweep args available.")

entity-swap eval is importable; coefficient sweep args available.


In [4]:
import torch

RELATIONS = [0, 3]  # None -> all 10 BATS relations (0..9)
SAMPLE_PAIRS_PER_RELATION = 50
RANDOM_STATE = 42
NEG_COEFFS = "-1,-2"  # source suppression factors
ADD_COEFFS = "4, 8"  # donor injection factors
LAYERS_BELOW = 0
LAYERS_ABOVE = 1

MODEL_NAME = "google/gemma-2-2b"
# Must match the transcoder set the graphs were built with (feature indices are set-specific).
TRANSCODER_SET = "mntss/clt-gemma-2-2b-426k"
BACKEND = "transformerlens"
DTYPE = "bfloat16"
DEVICE = "cuda" if shutil.which("nvidia-smi") else "cpu"
DTYPE_MAP = {"float32": torch.float32, "float16": torch.float16, "bfloat16": torch.bfloat16}

GRAPH_SRC_DIR = (
    REPO_ROOT
    / "summary_graphs"
    / "analogies"
    / "mntss"
    / "clt-gemma-2-2b-426k"
    / "entmax"
    / "alpha_0.50"
    / "node_0.02"
)
ANALOGIES_FILE = REPO_ROOT / "dataset" / "analogies" / "bats_analogies.txt"

WORK_ROOT = (Path("/content") if Path("/content").exists() else REPO_ROOT) / "entity_swap_full_sweep"
STAGED_DIR = WORK_ROOT / "numeric_ours"
OUTPUT_DIR = WORK_ROOT / "outputs"

assert GRAPH_SRC_DIR.is_dir(), f"missing labeled graph dir: {GRAPH_SRC_DIR}"
assert ANALOGIES_FILE.exists(), f"missing analogies file: {ANALOGIES_FILE}"
print("relations:", "all (0..9)" if RELATIONS is None else RELATIONS)
print("source factors:", NEG_COEFFS, "| donor factors:", ADD_COEFFS)
print("graph src:", GRAPH_SRC_DIR)
print("device:", DEVICE)

relations: [0, 3]
source factors: -1,-2 | donor factors: 4, 8
graph src: /home/tu/circuit_tracer_mod/summary_graphs/analogies/mntss/clt-gemma-2-2b-426k/entmax/alpha_0.50/node_0.02
device: cuda


In [5]:
# Stage NNN.sng.pt symlinks; the eval requires the .sng.pt suffix and exactly 100 numeric files.
from summarization.summarize import SummaryGraph

if STAGED_DIR.exists():
    shutil.rmtree(STAGED_DIR)
STAGED_DIR.mkdir(parents=True)

for idx in range(100):
    src = GRAPH_SRC_DIR / f"{idx:03d}.pt"
    assert src.exists(), f"missing labeled graph {src}"
    (STAGED_DIR / f"{idx:03d}.sng.pt").symlink_to(src)

staged = sorted(STAGED_DIR.glob("[0-9][0-9][0-9].sng.pt"))
assert len(staged) == 100, f"expected 100 staged graphs, found {len(staged)}"

probe = SummaryGraph.load(str(STAGED_DIR / "000.sng.pt"))
assert isinstance(probe.metadata.get("prompt"), str) and probe.metadata["prompt"], "000 missing metadata['prompt']"
assert probe.metadata.get("prompt_tokens"), "000 missing metadata['prompt_tokens']"
print("staged", len(staged), "graphs at", STAGED_DIR)
print("probe prompt:", probe.metadata["prompt"])

staged 100 graphs at /home/tu/circuit_tracer_mod/entity_swap_full_sweep/numeric_ours
probe prompt: <bos>The saying goes: abuja is to nigeria as amman is to


In [6]:
import os

# google/gemma-2-2b is a GATED model: you must accept its license on the HF model page and
# provide a token. mntss transcoders + the staged graphs are public, so this is the only auth step.
try:
    from google.colab import userdata  # type: ignore
except Exception:
    userdata = None

hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_API_KEY")
if hf_token is None and userdata is not None:
    # In Colab, add a secret named HF_TOKEN (Settings -> Secrets) and enable notebook access.
    try:
        hf_token = userdata.get("HF_TOKEN") or userdata.get("HUGGINGFACE_API_KEY")
    except Exception:
        hf_token = None

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    from huggingface_hub import login

    login(token=hf_token)
    print("Hugging Face token configured.")
else:
    print(
        "No HF token found. Set a Colab secret HF_TOKEN (or os.environ['HF_TOKEN']) and re-run; "
        "google/gemma-2-2b is gated and will 401 without it."
    )

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face token configured.


In [7]:
import argparse

from circuit_tracer import ReplacementModel
from eval.eval_entity_swap import run_entity_swap

print("loading replacement model once:", MODEL_NAME, "/", TRANSCODER_SET)
model = ReplacementModel.from_pretrained(
    MODEL_NAME,
    TRANSCODER_SET,
    backend=BACKEND,
    lazy_encoder=False,
    dtype=DTYPE_MAP[DTYPE],
    device=torch.device(DEVICE) if DEVICE else None,
)


args = argparse.Namespace(
    graph_dir=STAGED_DIR,
    analogies_file=ANALOGIES_FILE,
    negation_coefficients=NEG_COEFFS,
    addition_coefficients=ADD_COEFFS,
    relations=None if RELATIONS is None else ",".join(str(r) for r in RELATIONS),
    sample_pairs_per_relation=SAMPLE_PAIRS_PER_RELATION,
    pair_list=None,
    random_state=RANDOM_STATE,
    output_dir=OUTPUT_DIR,
    layers_below=LAYERS_BELOW,
    layers_above=LAYERS_ABOVE,
    mode="full",
)
print("=== running full entity swap over all relations ===", flush=True)
print("source factors:", NEG_COEFFS, "| donor factors:", ADD_COEFFS, flush=True)
run_entity_swap(model, args)
print("output dir:", OUTPUT_DIR)

loading replacement model once: google/gemma-2-2b / mntss/clt-gemma-2-2b-426k


Fetching 52 files:   0%|          | 0/52 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b into HookedTransformer
=== running full entity swap over all relations ===
source factors: -1,-2 | donor factors: 4, 8
output dir: /home/tu/circuit_tracer_mod/entity_swap_full_sweep/outputs


In [8]:
import pandas as pd

# Per-relation summary, keyed by (relation, source_factor, donor_factor).
summary_df = pd.read_csv(OUTPUT_DIR / "swap_summary.csv")
summary_df["delta_p_source"] = summary_df["mean_p_source_steered"] - summary_df["mean_p_source_clean"]
summary_df["delta_p_donor"] = summary_df["mean_p_donor_steered"] - summary_df["mean_p_donor_clean"]

report_cols = [
    "relation_idx",
    "relation_name",
    "source_factor",
    "donor_factor",
    "n_attempted",
    "n_eligible",
    "top1_hit_rate",
    "top1_hit_exact_rate",
    "top5_hit_rate",
    "success_rate",
    "eligible_success_rate",
    "mean_p_source_clean",
    "mean_p_source_steered",
    "delta_p_source",
    "mean_p_donor_clean",
    "mean_p_donor_steered",
    "delta_p_donor",
]
report_df = summary_df.sort_values(
    ["relation_idx", "source_factor", "donor_factor"]
).reset_index(drop=True)[report_cols]

# Aggregate across relations: mean metric per (source_factor, donor_factor) cell.
agg_cols = ["top1_hit_rate", "top5_hit_rate", "success_rate", "delta_p_source", "delta_p_donor"]
agg_df = (
    summary_df.groupby(["source_factor", "donor_factor"])[agg_cols]
    .mean()
    .reset_index()
    .sort_values(["source_factor", "donor_factor"])
    .reset_index(drop=True)
)

pd.set_option("display.float_format", lambda v: f"{v:.4f}")
print("=== aggregated across relations (mean per factor cell) ===")
display(agg_df)
print("=== per relation ===")
display(report_df)

=== aggregated across relations (mean per factor cell) ===


,source_factor,donor_factor,top1_hit_rate,top5_hit_rate,success_rate,delta_p_source,delta_p_donor
0,-2.0000,4.0000,0.3000,0.4800,0.3000,-0.5534,0.1566
1,-2.0000,8.0000,0.2800,0.5000,0.2800,-0.6246,0.1217
2,-1.0000,4.0000,0.3100,0.4600,0.3100,-0.5010,0.1752
3,-1.0000,8.0000,0.2900,0.4600,0.2900,-0.6057,0.1418


=== per relation ===


,relation_idx,relation_name,source_factor,donor_factor,n_attempted,n_eligible,top1_hit_rate,top1_hit_exact_rate,top5_hit_rate,success_rate,eligible_success_rate,mean_p_source_clean,mean_p_source_steered,delta_p_source,mean_p_donor_clean,mean_p_donor_steered,delta_p_donor
0,0,capital_country,-2.0000,4.0000,50,50,0.1600,0.1000,0.2800,0.1600,0.1600,0.7554,0.1813,-0.5741,0.0009,0.0953,0.0944
1,0,capital_country,-2.0000,8.0000,50,50,0.2000,0.1000,0.4000,0.2000,0.2000,0.7554,0.0808,-0.6746,0.0009,0.0826,0.0817
2,0,capital_country,-1.0000,4.0000,50,50,0.1600,0.1000,0.2800,0.1600,0.1600,0.7554,0.2419,-0.5135,0.0009,0.0971,0.0962
3,0,capital_country,-1.0000,8.0000,50,50,0.2200,0.1200,0.3600,0.2200,0.2200,0.7554,0.1025,-0.6529,0.0009,0.1017,0.1008
4,3,person_nationality,-2.0000,4.0000,50,50,0.4400,0.3000,0.6800,0.4400,0.4400,0.6063,0.0735,-0.5328,0.0078,0.2267,0.2188
5,3,person_nationality,-2.0000,8.0000,50,50,0.3600,0.2000,0.6000,0.3600,0.3600,0.6063,0.0316,-0.5747,0.0078,0.1695,0.1617
6,3,person_nationality,-1.0000,4.0000,50,50,0.4600,0.3200,0.6400,0.4600,0.4600,0.6063,0.1177,-0.4885,0.0078,0.2619,0.2541
7,3,person_nationality,-1.0000,8.0000,50,50,0.3600,0.2200,0.5600,0.3600,0.3600,0.6063,0.0479,-0.5584,0.0078,0.1906,0.1828


In [9]:
# Write a self-contained HTML report so it can be downloaded from Colab.
rate_cols = [
    "top1_hit_rate",
    "top1_hit_exact_rate",
    "top5_hit_rate",
    "success_rate",
    "eligible_success_rate",
    "mean_p_source_clean",
    "mean_p_source_steered",
    "delta_p_source",
    "mean_p_donor_clean",
    "mean_p_donor_steered",
    "delta_p_donor",
]
fmt = {col: "{:.4f}".format for col in rate_cols}
agg_fmt = {col: "{:.4f}".format for col in agg_cols}
agg_html = agg_df.to_html(index=False, formatters=agg_fmt, border=0)
table_html = report_df.to_html(index=False, formatters=fmt, border=0)

report_path = OUTPUT_DIR / "report.html"
report_path.write_text(
    """<!doctype html>
<html><head><meta charset=\"utf-8\"><title>Entity-swap full factor sweep</title>
<style>
body{font-family:system-ui,Arial,sans-serif;margin:2rem;color:#1a1a1a;}
h1{font-size:1.4rem;}h2{font-size:1.05rem;margin-top:1.5rem;}
table{border-collapse:collapse;margin-top:.5rem;font-size:.85rem;}
th,td{border:1px solid #ccc;padding:4px 8px;text-align:right;}
th{background:#f2f2f2;}td:first-child,th:first-child{text-align:left;}
code{background:#f2f2f2;padding:1px 4px;border-radius:3px;}
</style></head><body>
<h1>Entity-swap (full) factor sweep &mdash; all relations</h1>
<p>Full swap: source Output features suppressed at factors <code>%s</code>; donor Output features
injected at factors <code>%s</code>. Each relation uses %d sampled source&rarr;donor pairs
(random_state=%d). Transcoder set: <code>%s</code>. &Delta;p = steered &minus; clean.</p>
<h2>Aggregated across relations (mean per factor cell)</h2>
%s
<h2>Per relation</h2>
%s
</body></html>
"""
    % (NEG_COEFFS, ADD_COEFFS, SAMPLE_PAIRS_PER_RELATION, RANDOM_STATE, TRANSCODER_SET, agg_html, table_html),
    encoding="utf-8",
)
print("wrote report:", report_path)

wrote report: /home/tu/circuit_tracer_mod/entity_swap_full_sweep/outputs/report.html


In [10]:
# Verify: every relation has the full 3x3 factor grid over the same sampled pairs.
results = pd.read_csv(OUTPUT_DIR / "swap_results.csv")

expected_source_factors = {float(c) for c in NEG_COEFFS.split(",")}
expected_donor_factors = {float(c) for c in ADD_COEFFS.split(",")}
n_cells = len(expected_source_factors) * len(expected_donor_factors)

assert set(results["source_factor"].unique()) == expected_source_factors, results["source_factor"].unique()
assert set(results["donor_factor"].unique()) == expected_donor_factors, results["donor_factor"].unique()

for relation_idx, grp in results.groupby("relation_idx"):
    pairs = set(map(tuple, grp[["source_idx", "donor_idx"]].drop_duplicates().to_numpy().tolist()))
    cells = grp.groupby(["source_factor", "donor_factor"]).ngroups
    assert len(pairs) == SAMPLE_PAIRS_PER_RELATION, f"relation {relation_idx}: {len(pairs)} pairs"
    assert cells == n_cells, f"relation {relation_idx}: {cells} factor cells (expected {n_cells})"
    assert len(grp) == SAMPLE_PAIRS_PER_RELATION * n_cells, f"relation {relation_idx}: {len(grp)} rows"
    print(f"relation {relation_idx}: {len(pairs)} pairs x {cells} cells = {len(grp)} rows")

print(
    "verification passed;",
    results["relation_idx"].nunique(),
    "relations,",
    n_cells,
    "factor cells each,",
    len(results),
    "total rows",
)

relation 0: 50 pairs x 4 cells = 200 rows
relation 3: 50 pairs x 4 cells = 200 rows
verification passed; 2 relations, 4 factor cells each, 400 total rows
